In [1]:
import asyncio
import itertools

from datetime import datetime
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import LinearConstraint, milp, Bounds
from scipy.stats import multivariate_normal


In [2]:
class StateGenerator:

    def __init__(
        self,
        initial_condition: np.ndarray,
        turn_persistance: float,
        speed_err_std: float,
        tr_err_std: float
    ):
        """
        initial_condition: Initial state of the contact.
        turn_persistance: damping coefficient on turns. Defines how drawn out a turn can be.
        speed_err_std: standard deviation of speed error
        tr_err_std: standard deviation of turn rate in radians per unit time.
        """

        self.time_history = [0.0]
        self.state_history = [initial_condition]
        self.turn_persistance = turn_persistance
        self.speed_err_std = speed_err_std
        self.tr_err_std = tr_err_std
        self.last_update = None

    def update(self, time):
        """
        Defines a function to generate synthetic data representing slowly maneuvering ship contacts.
        """

        if self.last_update is None:
            dt = 0.1
        else:
            dt = (time - self.time_history[-1]).total_seconds()

        assert dt > 0

        x, y, speed, heading, turn_rate = self.state_history[-1]

        x_new = x + dt * np.cos(heading) * speed
        y_new = y + dt * np.sin(heading) * speed

        # Clamp speed to be positive.
        speed_new = np.max(
            [0.5, speed + np.random.normal(scale=self.speed_err_std)]
        )
        heading_new = heading + dt * turn_rate

        # Turning Logic:
        # If vehicle is driving straight, introduce small chance for maneuvering. Otherwise continue current turn.
        if np.abs(turn_rate) <= 1e-2:
            turn_chance = np.random.choice([0, 1], p=[.99, 0.01])
            turn_amount = np.random.choice(np.linspace(-np.deg2rad(5), np.deg2rad(5), 10))  # rad/s turn rate
            tr_new = turn_chance * turn_amount
        else:
            tr_new = self.turn_persistance * turn_rate + np.random.normal(scale=self.tr_err_std)
    
        self.time_history.append(time)
        self.state_history.append(np.asarray([x_new, y_new, speed_new, heading_new, tr_new]))


def system_model(x, dt):
    return np.asarray([
        x[0] + dt * np.cos(x[3]) * x[2],
        x[1] + dt * np.sin(x[3]) * x[2],
        x[2],
        x[3]
    ])


def J(x, dt):
    """
    Assumes state is ordered as x, y, speed, heading.
    """

    return np.array([
        [1, 0, dt * np.cos(x[3]), -dt * x[2] * np.sin(x[3])],
        [0, 1, dt * np.sin(x[3]),  dt * x[2] * np.cos(x[3])],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ])


def propagate_uncertainty(P, F, Q):
    """
    Returns updated value of P after prediction step.

    P: Estimation error covariance
    F: Linear transformation... Jacobian if system is non-linear
    Q: Process error covariance
    """
    return F @ P @ F.T + Q


def compute_gain(H, P, R):
    """
    Compute and return Kalman gain matrix K and innovation covariance S.

    H: Measurement model matrix
    P: Estimation error covariance
    R: Sensor error covariance
    """
    S = H @ P @ H.T + R
    K = P @ H.T @ np.linalg.inv(S)
    return K, S


def update_estimate(estimate, measurement, K, H):
    """
    Update distribution mean. One half of the Bayes update step.

    estimate: Predicted state at timestep k + 1.
    measurement: Measurement received at timestep k + 1.
    K: Kalman gain matrix.
    H: Measurement model matrix.

    Returns: update distribution mean and residual.
    """
    residual = (measurement - H @ estimate)
    update = estimate + K @ residual
    return update, residual


def update_uncertainty(P, R, H, K):
    """
    Update and return covariance of distribution. Second half of Bayes update step.

    P: Covariance of prediction at time k + 1
    R: Measurement error covariance
    H: Measurement model matrix
    K: Kalman gain matrix
    """
    I = np.eye(*P.shape)
    update = (I - K @ H) @ P @ (I - K @ H).T + K @ R @ K.T
    return update

In [3]:
# =========================
# Simulation Setup
# =========================

rng = np.random.default_rng()

## -------------- Grid Parameters ---------------------->
xmin = -2000
ymin = -2000
xmax = 2000
ymax = 2000
resolution = 6000


## -------------- Contact Parameters ------------------->
n_contacts = 3     # number of contacts


x = np.linspace(xmin, xmax, resolution)
y = np.linspace(ymin, ymax, resolution)


# --------------- Create Grid -------------------------->
X, Y = np.meshgrid(x, y)
coords = np.column_stack((X.ravel(), Y.ravel()))

# contacts will start on the boundary of the defined grid (bitwise OR)
boundary_mask = (
    (coords[:, 0] == xmin)
    | (coords[:, 0] == xmax)
    | (coords[:, 1] == ymin)
    | (coords[:, 1] == ymax)
)


# ------------- Initialize Contact Data ---------------->
# Initialize random starting positions around boundary of grid
init_pos = rng.choice(coords[boundary_mask], size=n_contacts)

# Initialize random start headings by choosing interior locations to point at
interior_pts = rng.choice(a=coords[~boundary_mask], size=n_contacts)
diff = interior_pts - init_pos

# arctan2 returns between -pi to pi, moving counter-clockwise from -x axis.
init_heading = np.pi / 2 - np.arctan2(*diff.T)  # rotate so 0 is up (north)
init_heading = np.mod(
    init_heading, 2*np.pi
)  # wrap around circle so angles are positive.

true_states = []
for c in range(n_contacts):
    state = np.asarray([
        init_pos[c, 0],                              # x
        init_pos[c, 1],                              # y
        np.abs(np.random.normal(loc=10, scale=3)),   # speed
        init_heading[c],                             # heading
        0                                            # turn rate
    ]).reshape(1, -1)

    true_states.append(state)

print(
    f"number of contacts: {len(true_states)} "
    f"\nstate vector size: {true_states[0].shape}"
)

number of contacts: 3 
state vector size: (1, 5)


In [4]:
# Each contact is a StateGenerator object.
# each sensor should essentially poll the ground truth from each contact (StateGenerator) at it...
# Generate truth as one async function at a fixed rate, greater than or equal to the f...
# Function 2...N-1 : Sensors, running at their individual rate. They pull most recent data tha...
# Function N + 1: The centralized tracker, which pulls data from sensor queues and performs da...

async def sensor(queue, rate, R, sensor_id):
    """
    An instance of a sensor loop.
    """

    R = np.diag([R**2] * 4)

    while True:

        time = datetime.now()
        for i, c in enumerate(contacts):
            c.update(time)
            true_state = c.state_history[-1][:-1]
            sensed_state = true_state + np.random.multivariate_normal([0] * 4, R)
            
            queue.put_nowait(
                {
                    "sensor_id": sensor_id,
                    "contact_id": i,
                    "timestamp": time,
                    "state": sensed_state,
                    "covariance": R
                }
            )

        await asyncio.sleep(1 / rate)

In [63]:
async def tracker(queue: asyncio.Queue, tracker_history: dict):
    """
    Receives and manages tracks from decentralized sensor tracker systems.

    queue: Asyncio queue that will receive tracks from sensors.
    tracker_history: The list which will be populated with the tracker's history
    """

    # Format of tracker_history
    # {
    #     sensor_id (int):
    #         {
    #             track_id (int):
    #             [
    #                 {time: 0, state: [x_0, y_0, vx_0, vy_0], covariance: P},
    #                 ...
    #                 {time: N, state: [x_N, y_N, vx_N, vy_N], covariance: P}
    #             ]
    #         }
    # }

    while True:
        output = await queue.get()

        sid = output.pop("sensor_id")
        tid = output.pop("contact_id")

        if sid not in tracker_history:
            tracker_history[sid] = {}

        if tid not in tracker_history[sid]:
            tracker_history[sid][tid] = []

        tracker_history[sid][tid].append(output)

        # Form pools of candidate tracks and their covariances, one for each sensor.
        n_sensors = len(tracker_history)
        n_tracks_per_sensor = [len(tracker_history[sensor_id]) for sensor_id in tracker_history]

        if n_sensors < 2:
            continue

        # This forces the assumption of complete associations for the prototype. 
        if len(set(n_tracks_per_sensor)) != 1:  
            continue

        # Lists of lists, where inner list is a sensors contributing track states and covariances. 
        candidate_covs = [[] for _ in range(n_sensors)]
        candidate_states = [[] for _ in range(n_sensors)]

        # Fill each sensor's contributing pool of track states and covariances. 
        for i, sensor_id in enumerate(tracker_history):
            for track_id in tracker_history[sensor_id]:

                # Get this track's state over time
                sensor_history = tracker_history[sensor_id][track_id]

                # Get the most recent state and covariance:
                state = sensor_history[-1]["state"]
                covariance = sensor_history[-1]["covariance"]

                # Append to candidate pool for this iteration.
                candidate_states[i].append(state)
                candidate_covs[i].append(covariance)

        # Form the tuples of potentially associated states and covariances from the cartesian product of contributing lists:
        candidates = zip(
            itertools.product(*candidate_states),
            itertools.product(*candidate_covs)
        )

        # Compute cost of each candidate hypothesis. 
        costs = []
        for x, P in candidates:

            # Compute likelihood covariance - assumes error between sensors is mutually independent
            P_blocks = [
                [
                    P[0] + P[i] if i == j else P[0]
                    for i in range(1, n_sensors)
                ]
                for j in range(1, n_sensors)
            ]

            P_delta = np.block(P_blocks)

            # assemble likelihood input vector... (n_sensors - 1, n_states)
            x = np.asarray(x)
            x_hat = x[1:] - x[0]
            x_hat = x_hat.flatten()

            # Define the PDF
            mvn = multivariate_normal(
                mean=np.zeros_like(x_hat),
                cov=P_delta
            )

            # compute cost as negative log likelihood.
            c = -mvn.logpdf(x_hat)

            # append cost to lists.
            costs.append(c)


        # ----------- Perform integer linear optimization ----------

        # form constraint matrix A
        A = []
        hypothesis_indexes = list(itertools.product(*[np.arange(len(r)) for r in tracker_history.values()]))

        for i in range(n_sensors):
            n_tracks = len(tracker_history[i])

            for j in range(n_tracks):
                row = [1 if h[i] == j else 0 for h in hypothesis_indexes]
                A.append(row)
                
        # define scipy LinearConstraint from constraint matrix and bounds Ax = 1.
        problem_constraints = LinearConstraint(A, lb=np.ones(len(A)), ub=np.ones(len(A)))
        decision_boundary = Bounds(lb=0, ub=1)

        res = milp(c=costs, integrality=1, bounds=decision_boundary, constraints=problem_constraints)
        truth_mask = res.x == 1
        matches = np.asarray(hypothesis_indexes)[truth_mask]
        print(matches)

In [64]:
async def main(data_return, runtime, sensor_rates, sensor_std_dev):
    """
    Main simulation Loop
    """

    queue = asyncio.Queue()

    try:
        async with asyncio.timeout(runtime):
            await asyncio.gather(
                tracker(queue, data_return),
                *(sensor(queue, r, s, i) for r, s, i in zip(
                        sensor_rates,
                        sensor_std_dev,
                        range(3))
                 )
            )
            
    except TimeoutError:
        print("Simulation Complete")

In [65]:
stop_time = 1
turn_persistance = 0.9
speed_err_std = 0.5
tr_err_std = np.deg2rad(0.05)

contacts = [
    StateGenerator(x_0.squeeze(), turn_persistance, speed_err_std, tr_err_std) 
    for x_0 in true_states
]

# sensor parameters
sensor_rates = [50, 50, 100]
std_dev = [7, 2, 1]

data = {}
await main(data, stop_time, sensor_rates, std_dev)

[[0 0]
 [1 1]
 [2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]]
[[0 0 0]
 [1 1 1]
 [2 2 2]

In [50]:
mask = np.asarray([0, 1, 0, 1]) == 1
-

TypeError: only integer scalar arrays can be converted to a scalar index

In [9]:
[np.arange(len(r)) for r in data.values()]

[array([0, 1, 2]), array([0, 1, 2]), array([0, 1, 2])]